In [ ]:
# 1. Create the final directory
!mkdir -p /content/unlabeled_images

# 2. Unzip to a temporary folder
!unzip -q /content/unlabeled_images.zip -d /content/temp

# 3. Find all real files (ignoring the Mac junk) and move them to our folder
!find /content/temp -type f ! -path "*/__MACOSX/*" -exec mv {} /content/unlabeled_images/ \;

# 4. Clean up the mess
!rm -rf /content/temp

# 5. Count the files to prove it worked!
!echo "Total images perfectly extracted:"
!ls -1 /content/unlabeled_images | wc -l

In [ ]:
!pip -q install --upgrade unitlab opencv-python scikit-image albumentations tqdm

In [ ]:
!mkdir -p /root/.unitlab

In [ ]:
pip install --upgrade unitlab

In [ ]:
from unitlab import UnitlabClient
import os

API_KEY = ""

os.makedirs("/root/.unitlab", exist_ok=True)
with open("/root/.unitlab/credentials", "w") as f:
    f.write("[default]\n")
    f.write(f"api_key={API_KEY}\n")
    f.write("api_url=https://api.unitlab.ai\n")

from unitlab import UnitlabClient
client = UnitlabClient(API_KEY)  # ✅ this SDK wants api_key as positional too
print("✅ client ready")

In [ ]:
ds = client.datasets()
print(type(ds))
print(ds)

In [ ]:
DATASET_ID = "dabbdf31-47db-4c22-929f-d71ec3c33dbf"

In [ ]:
from unitlab import UnitlabClient
from unitlab import exceptions as ul_exceptions

print("✅ client initialized")

# DATASET_ID already set from datasets[..]["pk"]
print("Using dataset:", DATASET_ID)

for split in ["train", "valid", "validation", "test"]:
    try:
        print(f"\nAttempting download for split: {split}")
        client.dataset_download(DATASET_ID, "COCO", split)
        print(f"✅ Downloaded annotations for split: {split}")
    except Exception as e:
        print(f"⚠️ Could not download split '{split}': {e}")

# Download raw image files once (this is not split-specific)
try:
    client.dataset_download_files(DATASET_ID)
    print("✅ Downloaded dataset files (images)")
except Exception as e:
    print("⚠️ Could not download files:", e)

In [ ]:
import inspect
import nest_asyncio
nest_asyncio.apply()

from unitlab import UnitlabClient
DATASET_ID = "dabbdf31-47db-4c22-929f-d71ec3c33dbf"

# These already work:
client.dataset_download(DATASET_ID, "COCO", "train")
client.dataset_download(DATASET_ID, "COCO", "validation")
client.dataset_download(DATASET_ID, "COCO", "test")
# Now handle dataset_download_files across SDK versions
fn = client.dataset_download_files

result = fn(DATASET_ID)

# If it returned an awaitable, await it
if inspect.isawaitable(result):
    await result
    print("✅ dataset_download_files awaited successfully")
else:
    # If not awaitable, it already ran (or failed internally)
    print("ℹ️ dataset_download_files returned non-awaitable:", type(result))
    print("✅ if no error above, files download is complete (or not needed)")

In [ ]:
import os, json, collections
import numpy as np
import cv2

from pycocotools import mask as maskUtils

train_json = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_tra.json"
img_dir    = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"

with open(train_json, "r") as f:
    coco = json.load(f)

# build ann index
ann_index = {}
for a in coco["annotations"]:
    ann_index.setdefault(a["image_id"], []).append(a)

size_counter_seg = collections.Counter()
size_counter_decoded_raw = collections.Counter()
size_counter_decoded_fixed = collections.Counter()

def decode_raw(seg):
    """Decode using seg['size'] interpreted as (H,W) directly (no swaps, no flips)."""
    h0, w0 = seg["size"]  # treating as H,W
    # If counts is list: use frPyObjects
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode(seg)
    if m.ndim == 3:
        m = m[:,:,0]
    return m.astype(np.uint8)

def decode_fixed(seg):
    """Your known fix: seg['size'] stored [W,H] -> swap, then flip + rot90 CCW."""
    w0, h0 = seg["size"]  # stored W,H
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode({"size":[h0,w0], "counts": seg["counts"]})
    if m.ndim == 3:
        m = m[:,:,0]
    m = m.astype(np.uint8)
    m = cv2.flip(m, 1)
    m = np.rot90(m, 1).copy()
    return m

# scan everything
for im in coco["images"]:
    anns = ann_index.get(im["id"], [])
    for a in anns:
        seg = a.get("segmentation", None)
        if not (isinstance(seg, dict) and "size" in seg and "counts" in seg):
            continue

        size_counter_seg[tuple(seg["size"])] += 1

        try:
            m_raw = decode_raw(seg)
            size_counter_decoded_raw[m_raw.shape] += 1
        except Exception:
            pass

        try:
            m_fix = decode_fixed(seg)
            size_counter_decoded_fixed[m_fix.shape] += 1
        except Exception:
            pass

print("Unique seg['size'] values (stored):")
for k,v in size_counter_seg.most_common(20):
    print("  ", k, "count:", v)

print("\nUnique decoded RAW shapes (treat seg['size'] as H,W):")
for k,v in size_counter_decoded_raw.most_common(20):
    print("  ", k, "count:", v)

print("\nUnique decoded FIXED shapes (swap + flip + rot90ccw):")
for k,v in size_counter_decoded_fixed.most_common(20):
    print("  ", k, "count:", v)


In [ ]:
import os, json, math
import numpy as np
import cv2
import matplotlib.pyplot as plt

from pycocotools import mask as maskUtils

train_json = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_tra.json"
img_dir    = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"

PER_PAGE = 8  # masks per page

with open(train_json, "r") as f:
    coco = json.load(f)

# index annotations
ann_index = {}
for a in coco["annotations"]:
    ann_index.setdefault(a["image_id"], []).append(a)

def decode_fixed(seg):
    """Your dataset fix: seg['size'] stored [W,H] -> swap, then flip + rot90 CCW."""
    w0, h0 = seg["size"]  # stored W,H
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode({"size":[h0,w0], "counts": seg["counts"]})
    if m.ndim == 3:
        m = m[:,:,0]
    m = m.astype(np.uint8)
    m = cv2.flip(m, 1)
    m = np.rot90(m, 1).copy()
    return m  # 0/1

# keep only images that exist and have at least 1 segmentation
items = []
for im in coco["images"]:
    fp = os.path.join(img_dir, im["file_name"])
    if not os.path.isfile(fp):
        continue
    anns = ann_index.get(im["id"], [])
    ok = any(isinstance(a.get("segmentation"), dict) and "size" in a["segmentation"] for a in anns)
    if ok:
        items.append(im)

print("Images with file + segmentation:", len(items))

pages = math.ceil(len(items) / PER_PAGE)

for p in range(pages):
    batch = items[p*PER_PAGE:(p+1)*PER_PAGE]

    plt.figure(figsize=(18, 10))
    for i, im in enumerate(batch, start=1):
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        Himg, Wimg = img.shape[:2]

        anns = ann_index[im["id"]]

        # union all masks in their native fixed space (no resizing)
        native = None
        stored_sizes = []
                # union all masks on a common canvas (image size)
        native = np.zeros((Himg, Wimg), dtype=np.uint8)
        stored_sizes = []
        bad_shapes = []

        for a in anns:
            seg = a.get("segmentation", None)
            if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
                continue

            stored_sizes.append(tuple(seg["size"]))
            m = decode_fixed(seg)  # 0/1

            # force to image canvas
            if m.shape != (Himg, Wimg):
                bad_shapes.append(m.shape)
                m = cv2.resize(m, (Wimg, Himg), interpolation=cv2.INTER_NEAREST)

            native = np.maximum(native, m)


        if native is None:
            native = np.zeros((1,1), dtype=np.uint8)

        Hm, Wm = native.shape

        # Plot mask only
        plt.subplot(2,4,i)
        plt.title(f"{im['file_name']}\nimg:{Himg}x{Wimg}  coco:{im.get('height','?')}x{im.get('width','?')}\nseg sizes:{set(stored_sizes)}  mask:{Hm}x{Wm}",
                  fontsize=7)
        plt.imshow(native*255, cmap="gray")
        plt.axis("off")

    plt.suptitle(f"Masks only (native, no resize) | page {p+1}/{pages}", fontsize=14)
    plt.show()


In [ ]:
img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
overlay = img_color.copy()
overlay[native == 1] = [0,0,255]  # red

blend = cv2.addWeighted(img_color, 0.7, overlay, 0.3, 0)
plt.imshow(blend[:,:,::-1])
plt.axis("off")


In [ ]:
def decode_raw(seg):
    # seg["size"] stored as [W,H] in your dataset → use [H,W] for pycocotools
    w0, h0 = seg["size"]
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode({"size":[h0,w0], "counts": seg["counts"]})
    if m.ndim == 3:
        m = m[:, :, 0]
    return m.astype(np.uint8)  # 0/1


In [ ]:
def apply_tf(m, rot_k=0, flip=None):
    # rot_k: 0,1,2,3 (CCW)
    if flip == "h":
        m = cv2.flip(m, 1)
    elif flip == "v":
        m = cv2.flip(m, 0)
    elif flip == "hv":
        m = cv2.flip(m, -1)
    m = np.rot90(m, rot_k).copy()
    return m

def show_8_overlays(img_gray, m_raw):
    H, W = img_gray.shape
    img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

    configs = [
        ("r0", 0, None), ("r90", 1, None), ("r180", 2, None), ("r270", 3, None),
        ("h+r0", 0, "h"), ("h+r90", 1, "h"), ("h+r180", 2, "h"), ("h+r270", 3, "h"),
        # if none look right, swap "h" with "v" and re-run
    ]

    plt.figure(figsize=(18, 9))
    for i,(name,rk,fl) in enumerate(configs, 1):
        m = apply_tf(m_raw, rk, fl)

        # resize to image for viewing ONLY (nearest)
        if m.shape != (H, W):
            m_view = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        else:
            m_view = m

        overlay = img_rgb.copy()
        overlay[m_view > 0] = (255, 0, 0)  # red mask pixels

        blend = cv2.addWeighted(img_rgb, 0.75, overlay, 0.25, 0)

        plt.subplot(2,4,i)
        plt.title(f"{name}\nmask:{m.shape}")
        plt.imshow(blend)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
seg = ann_index[im["id"]][0]["segmentation"]
m_raw = decode_raw(seg)
show_8_overlays(img, m_raw)


In [ ]:
m = decode_raw(seg)
m = cv2.flip(m, 1)          # horizontal flip
m = np.rot90(m, 1).copy()   # 90° CCW


In [ ]:
def overlay(img_gray, m):
    H,W = img_gray.shape
    if m.shape != (H,W):
        m = cv2.resize(m, (W,H), interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    ov = rgb.copy()
    ov[m>0] = (255,0,0)
    return cv2.addWeighted(rgb, 0.75, ov, 0.25, 0)

def try_4(img_gray, m_raw):
    variants = {
        "A: hflip -> rot90CCW": lambda x: np.rot90(cv2.flip(x,1), 1).copy(),
        "B: rot90CCW -> hflip": lambda x: cv2.flip(np.rot90(x,1).copy(), 1),
        "C: vflip -> rot90CCW": lambda x: np.rot90(cv2.flip(x,0), 1).copy(),
        "D: rot90CCW -> vflip": lambda x: cv2.flip(np.rot90(x,1).copy(), 0),
    }
    plt.figure(figsize=(16,8))
    for i,(name,fn) in enumerate(variants.items(),1):
        plt.subplot(2,2,i)
        plt.title(name)
        plt.imshow(overlay(img_gray, fn(m_raw)))
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
seg = ann_index[im["id"]][0]["segmentation"]
m_raw = decode_raw(seg)
try_4(img, m_raw)


In [ ]:
def decode_fixed_bbox(seg, bbox, Himg, Wimg):
    # 1) decode raw (fix size order only)
    w0, h0 = seg["size"]  # stored W,H
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode({"size":[h0,w0], "counts": seg["counts"]})
    if m.ndim == 3:
        m = m[:, :, 0]
    m = m.astype(np.uint8)

    # 2) apply the correct transform (A): hflip -> rot90 CCW
    m = cv2.flip(m, 1)
    m = np.rot90(m, 1).copy()

    # 3) paste into full image canvas using bbox
    x, y, w, h = bbox
    x0, y0 = int(round(x)), int(round(y))
    w0b, h0b = int(round(w)), int(round(h))

    # resize mask to bbox size (this is the ONLY scaling that should happen)
    m = cv2.resize(m, (w0b, h0b), interpolation=cv2.INTER_NEAREST)

    full = np.zeros((Himg, Wimg), dtype=np.uint8)

    # clip to image bounds
    x1 = max(0, x0); y1 = max(0, y0)
    x2 = min(Wimg, x0 + w0b); y2 = min(Himg, y0 + h0b)

    mx1 = x1 - x0; my1 = y1 - y0
    mx2 = mx1 + (x2 - x1); my2 = my1 + (y2 - y1)

    if x2 > x1 and y2 > y1:
        full[y1:y2, x1:x2] = m[my1:my2, mx1:mx2]

    return full  # full-frame 0/1


In [ ]:
native = np.zeros((Himg, Wimg), dtype=np.uint8)
for a in anns:
    seg = a.get("segmentation", None)
    if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
        continue
    bbox = a.get("bbox", None)
    if bbox is None:
        continue

    m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
    native = np.maximum(native, m_full)


In [ ]:
# --- after your union loop ---
print("native:", native.shape, "sum:", int(native.sum()), "max:", int(native.max()))

# 1) mask only
plt.figure(figsize=(6,6))
plt.title("Union mask (native)")
plt.imshow(native*255, cmap="gray")
plt.axis("off")
plt.show()

# 2) overlay on the image
img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  # <-- make sure `img` is loaded earlier
overlay = img_rgb.copy()
overlay[native > 0] = (255, 0, 0)  # red where mask is 1
blend = cv2.addWeighted(img_rgb, 0.75, overlay, 0.25, 0)

plt.figure(figsize=(6,6))
plt.title("Overlay")
plt.imshow(blend)
plt.axis("off")
plt.show()


In [ ]:
img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6,6))
plt.title("Original Image")
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()


In [ ]:
import math

PER_PAGE = 8  # images per page

# keep only images that exist (no segmentation filtering now)
items = []
for im in coco["images"]:
    fp = os.path.join(img_dir, im["file_name"])
    if os.path.isfile(fp):
        items.append(im)

print("Total images found:", len(items))

# limit to first 50
items = items[:50]

pages = math.ceil(len(items) / PER_PAGE)

for p in range(pages):
    batch = items[p*PER_PAGE:(p+1)*PER_PAGE]

    plt.figure(figsize=(18,10))
    for i, im in enumerate(batch, start=1):
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)

        plt.subplot(2,4,i)
        plt.title(im["file_name"], fontsize=8)
        plt.imshow(img, cmap="gray")
        plt.axis("off")

    plt.suptitle(f"Training Images Only | page {p+1}/{pages}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
import os, json, math
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pycocotools import mask as maskUtils

PER_PAGE = 8
N_SHOW = 50

def decode_fixed_bbox(seg, bbox, Himg, Wimg):
    # 1) decode raw (fix size order only)
    w0, h0 = seg["size"]  # stored W,H
    if isinstance(seg["counts"], list):
        rles = maskUtils.frPyObjects({"size":[h0,w0], "counts": seg["counts"]}, h0, w0)
        rle = maskUtils.merge(rles) if isinstance(rles, list) else rles
        m = maskUtils.decode(rle)
    else:
        m = maskUtils.decode({"size":[h0,w0], "counts": seg["counts"]})
    if m.ndim == 3:
        m = m[:, :, 0]
    m = m.astype(np.uint8)

    # 2) transform (A): hflip -> rot90 CCW
    m = cv2.flip(m, 1)
    m = np.rot90(m, 1).copy()

    # 3) paste into full image canvas using bbox
    x, y, w, h = bbox
    x0, y0 = int(round(x)), int(round(y))
    w0b, h0b = int(round(w)), int(round(h))

    # resize mask to bbox size (ONLY scaling that should happen)
    if w0b <= 0 or h0b <= 0:
        return np.zeros((Himg, Wimg), dtype=np.uint8)

    m = cv2.resize(m, (w0b, h0b), interpolation=cv2.INTER_NEAREST)

    full = np.zeros((Himg, Wimg), dtype=np.uint8)

    # clip to image bounds
    x1 = max(0, x0); y1 = max(0, y0)
    x2 = min(Wimg, x0 + w0b); y2 = min(Himg, y0 + h0b)

    mx1 = x1 - x0; my1 = y1 - y0
    mx2 = mx1 + (x2 - x1); my2 = my1 + (y2 - y1)

    if x2 > x1 and y2 > y1:
        full[y1:y2, x1:x2] = m[my1:my2, mx1:mx2]

    return full  # 0/1 full-frame


# ---- load coco (you already have this in your notebook; keep if needed) ----
with open(train_json, "r") as f:
    coco = json.load(f)

ann_index = {}
for a in coco["annotations"]:
    ann_index.setdefault(a["image_id"], []).append(a)

# keep images that exist AND have at least 1 annotation with bbox+seg
items = []
for im in coco["images"]:
    fp = os.path.join(img_dir, im["file_name"])
    if not os.path.isfile(fp):
        continue
    anns = ann_index.get(im["id"], [])
    ok = any(
        isinstance(a.get("segmentation"), dict) and "size" in a["segmentation"] and
        a.get("bbox") is not None
        for a in anns
    )
    if ok:
        items.append(im)

print("Images with file + seg+bbox:", len(items))

items = items[:min(N_SHOW, len(items))]
pages = math.ceil(len(items) / PER_PAGE)

for p in range(pages):
    batch = items[p*PER_PAGE:(p+1)*PER_PAGE]

    # 3 rows: image / mask / overlay
    plt.figure(figsize=(18, 16))

    for i, im in enumerate(batch, start=1):
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        Himg, Wimg = img.shape[:2]

        anns = ann_index.get(im["id"], [])

        # union mask using your decode_fixed_bbox
        native = np.zeros((Himg, Wimg), dtype=np.uint8)
        for a in anns:
            seg = a.get("segmentation", None)
            bbox = a.get("bbox", None)
            if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
                continue
            if bbox is None:
                continue
            m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
            native = np.maximum(native, m_full)

        # --- Row 1: image only ---
        ax1 = plt.subplot(3, PER_PAGE, i)
        ax1.set_title(im["file_name"], fontsize=8)
        ax1.imshow(img, cmap="gray")
        ax1.axis("off")

        # --- Row 2: mask only ---
        ax2 = plt.subplot(3, PER_PAGE, PER_PAGE + i)
        ax2.set_title(f"mask sum={int(native.sum())}", fontsize=8)
        ax2.imshow(native * 255, cmap="gray")
        ax2.axis("off")

        # --- Row 3: overlay ---
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        overlay = img_rgb.copy()
        overlay[native > 0] = (255, 0, 0)
        blend = cv2.addWeighted(img_rgb, 0.75, overlay, 0.25, 0)

        ax3 = plt.subplot(3, PER_PAGE, 2*PER_PAGE + i)
        ax3.set_title("overlay", fontsize=8)
        ax3.imshow(blend)
        ax3.axis("off")

    plt.suptitle(f"Images + bbox-pasted masks | page {p+1}/{pages}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import cv2
import numpy as np

mask_out_dir = "/content/train_masks"
os.makedirs(mask_out_dir, exist_ok=True)

saved = 0

for im in items:   # items already filtered images
    fp = os.path.join(img_dir, im["file_name"])
    img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue

    Himg, Wimg = img.shape[:2]
    anns = ann_index.get(im["id"], [])

    native = np.zeros((Himg, Wimg), dtype=np.uint8)

    for a in anns:
        seg = a.get("segmentation", None)
        bbox = a.get("bbox", None)

        if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
            continue
        if bbox is None:
            continue

        m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
        native = np.maximum(native, m_full)

    # convert 0/1 → 0/255
    mask_uint8 = (native * 255).astype(np.uint8)

    out_path = os.path.join(mask_out_dir, im["file_name"])
    cv2.imwrite(out_path, mask_uint8)

    saved += 1

print("Saved masks:", saved)
print("Mask folder:", mask_out_dir)


In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

def index_annotations(coco):
    ann_index = {}
    for a in coco.get("annotations", []):
        ann_index.setdefault(a["image_id"], []).append(a)
    return ann_index

def collect_items(coco, img_dir, ann_index=None, require_ann=True):
    items = []
    for im in coco["images"]:
        fp = os.path.join(img_dir, im["file_name"])
        if not os.path.isfile(fp):
            continue

        if require_ann:
            anns = (ann_index or {}).get(im["id"], [])
            ok = any(
                isinstance(a.get("segmentation"), dict) and "size" in a["segmentation"]
                and a.get("bbox") is not None
                for a in anns
            )
            if not ok:
                continue

        items.append(im)
    return items

def save_union_masks(coco_json_path, img_dir, mask_out_dir, decode_fixed_bbox, require_ann=True):
    os.makedirs(mask_out_dir, exist_ok=True)

    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    ann_index = index_annotations(coco)
    items = collect_items(coco, img_dir, ann_index=ann_index, require_ann=require_ann)

    saved = 0
    for im in items:
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        Himg, Wimg = img.shape[:2]
        anns = ann_index.get(im["id"], [])

        native = np.zeros((Himg, Wimg), dtype=np.uint8)
        for a in anns:
            seg = a.get("segmentation", None)
            bbox = a.get("bbox", None)
            if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
                continue
            if bbox is None:
                continue

            m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
            native = np.maximum(native, m_full)

        mask_uint8 = (native * 255).astype(np.uint8)

        out_path = os.path.join(mask_out_dir, im["file_name"])
        os.makedirs(os.path.dirname(out_path), exist_ok=True)  # in case file_name has subfolders
        cv2.imwrite(out_path, mask_uint8)
        saved += 1

    print("JSON:", coco_json_path)
    print("Images found:", len(items))
    print("Masks saved:", saved)
    print("Mask dir:", mask_out_dir)

    return coco, ann_index, items


In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

def index_annotations(coco):
    ann_index = {}
    for a in coco.get("annotations", []):
        ann_index.setdefault(a["image_id"], []).append(a)
    return ann_index

def collect_items(coco, img_dir, ann_index=None, require_ann=True):
    items = []
    for im in coco["images"]:
        fp = os.path.join(img_dir, im["file_name"])
        if not os.path.isfile(fp):
            continue

        if require_ann:
            anns = (ann_index or {}).get(im["id"], [])
            ok = any(
                isinstance(a.get("segmentation"), dict) and "size" in a["segmentation"]
                and a.get("bbox") is not None
                for a in anns
            )
            if not ok:
                continue

        items.append(im)
    return items

def save_union_masks(coco_json_path, img_dir, mask_out_dir, decode_fixed_bbox, require_ann=True):
    os.makedirs(mask_out_dir, exist_ok=True)

    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    ann_index = index_annotations(coco)
    items = collect_items(coco, img_dir, ann_index=ann_index, require_ann=require_ann)

    saved = 0
    for im in items:
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        Himg, Wimg = img.shape[:2]
        anns = ann_index.get(im["id"], [])

        native = np.zeros((Himg, Wimg), dtype=np.uint8)
        for a in anns:
            seg = a.get("segmentation", None)
            bbox = a.get("bbox", None)
            if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
                continue
            if bbox is None:
                continue

            m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
            native = np.maximum(native, m_full)

        mask_uint8 = (native * 255).astype(np.uint8)

        out_path = os.path.join(mask_out_dir, im["file_name"])
        os.makedirs(os.path.dirname(out_path), exist_ok=True)  # in case file_name has subfolders
        cv2.imwrite(out_path, mask_uint8)
        saved += 1

    print("JSON:", coco_json_path)
    print("Images found:", len(items))
    print("Masks saved:", saved)
    print("Mask dir:", mask_out_dir)

    return coco, ann_index, items


In [ ]:
val_json  = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_val.json"
test_json = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_tes.json"

val_masks_dir  = "/content/val_masks"
test_masks_dir = "/content/test_masks"

# ---------- helpers ----------
def index_annotations(coco):
    ann_index = {}
    for a in coco.get("annotations", []):
        ann_index.setdefault(a["image_id"], []).append(a)
    return ann_index

def collect_items(coco, img_dir, ann_index=None, require_ann=True):
    items = []
    for im in coco["images"]:
        fp = os.path.join(img_dir, im["file_name"])
        if not os.path.isfile(fp):
            continue

        if require_ann:
            anns = (ann_index or {}).get(im["id"], [])
            ok = any(
                isinstance(a.get("segmentation"), dict) and "size" in a["segmentation"]
                and a.get("bbox") is not None
                for a in anns
            )
            if not ok:
                continue

        items.append(im)
    return items

def save_union_masks(coco_json_path, img_dir, mask_out_dir, decode_fixed_bbox, require_ann=True):
    os.makedirs(mask_out_dir, exist_ok=True)

    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    ann_index = index_annotations(coco)
    items = collect_items(coco, img_dir, ann_index=ann_index, require_ann=require_ann)

    saved = 0
    for im in items:
        fp = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        Himg, Wimg = img.shape[:2]
        anns = ann_index.get(im["id"], [])

        native = np.zeros((Himg, Wimg), dtype=np.uint8)
        for a in anns:
            seg = a.get("segmentation", None)
            bbox = a.get("bbox", None)
            if not (isinstance(seg, dict) and "counts" in seg and "size" in seg):
                continue
            if bbox is None:
                continue

            m_full = decode_fixed_bbox(seg, bbox, Himg, Wimg)
            native = np.maximum(native, m_full)

        mask_uint8 = (native * 255).astype(np.uint8)

        out_path = os.path.join(mask_out_dir, im["file_name"])
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        cv2.imwrite(out_path, mask_uint8)
        saved += 1

    print("\nJSON:", coco_json_path)
    print("Images found:", len(items))
    print("Masks saved:", saved)
    print("Mask dir:", mask_out_dir)

    return coco, ann_index, items

# ---------- run VAL ----------
coco_val, ann_val, items_val = save_union_masks(val_json, img_dir, val_masks_dir, decode_fixed_bbox, require_ann=True)

# ---------- run TEST ----------
# If the test json has annotations, this will save them.
# If it has NO annotations (common), items_test will be 0.
coco_test, ann_test, items_test = save_union_masks(test_json, img_dir, test_masks_dir, decode_fixed_bbox, require_ann=True)

# If test s


In [ ]:
import random
import matplotlib.pyplot as plt

def preview_pairs(items, img_dir, mask_dir, n=12):
    sample = items[:]
    random.shuffle(sample)
    sample = sample[:min(n, len(sample))]

    cols = 4
    rows = int(np.ceil(len(sample) / cols))
    plt.figure(figsize=(16, 4*rows))

    for i, im in enumerate(sample, start=1):
        fp_img  = os.path.join(img_dir, im["file_name"])
        fp_mask = os.path.join(mask_dir, im["file_name"])

        img  = cv2.imread(fp_img, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(fp_mask, cv2.IMREAD_GRAYSCALE)

        plt.subplot(rows, cols*2, (i-1)*2 + 1)
        plt.title(f"IMG: {im['file_name']}", fontsize=8)
        plt.imshow(img, cmap="gray"); plt.axis("off")

        plt.subplot(rows, cols*2, (i-1)*2 + 2)
        plt.title(f"MASK sum={int(mask.sum()) if mask is not None else 'None'}", fontsize=8)
        plt.imshow(mask, cmap="gray"); plt.axis("off")

    plt.tight_layout()
    plt.show()

print("\nPreview VAL:")
preview_pairs(items_val, img_dir, val_masks_dir, n=12)

print("\nPreview TEST:")
preview_pairs(items_test, img_dir, test_masks_dir, n=12)


In [ ]:
!pip install segmentation-models-pytorch albumentations opencv-python


In [ ]:
import os, glob, time, random
import numpy as np
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp


In [ ]:
# Images (your dataset folder)
img_dir = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"

# Masks (generated earlier)
train_msk_dir = "/content/train_masks"
val_msk_dir   = "/content/val_masks"
test_msk_dir  = "/content/test_masks"

# Which split JSONs (only needed if you want to filter exact images per split)
train_json = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_tra.json"
val_json   = "/content/coco-dabbdf31-47db-4c22-929f-d71ec3c33dbf-v0.3_val.json"

# Device (CUDA / MPS / CPU)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

if device == "mps":
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


In [ ]:
def debug_pairs(img_dir, mask_dir, n=10):
    imgs = sorted([p for p in glob.glob(os.path.join(img_dir, "*")) if os.path.isfile(p)])
    msks = sorted([p for p in glob.glob(os.path.join(mask_dir, "*")) if os.path.isfile(p)])

    img_names = set(os.path.basename(p) for p in imgs)
    msk_names = set(os.path.basename(p) for p in msks)

    inter = sorted(list(img_names & msk_names))

    print("img_dir:", img_dir, "count:", len(imgs))
    print("mask_dir:", mask_dir, "count:", len(msks))
    print("pairs:", len(inter))

    if len(imgs) > 0:
        print("\nSample image files:")
        for p in imgs[:n]:
            print("  ", os.path.basename(p))

    if len(msks) > 0:
        print("\nSample mask files:")
        for p in msks[:n]:
            print("  ", os.path.basename(p))

    if len(inter) > 0:
        print("\nSample matched pairs:")
        for name in inter[:n]:
            print("  ", name)

debug_pairs(img_dir, train_msk_dir)


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset

class VeinSegDatasetSDF(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.transform = transform

        img_paths = sorted([p for p in glob.glob(os.path.join(img_dir, "*")) if os.path.isfile(p)])
        pairs = []
        for p in img_paths:
            name = os.path.basename(p)
            mp = os.path.join(mask_dir, name)
            if os.path.isfile(mp):
                pairs.append((p, mp))
        if len(pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found.\nimg_dir={img_dir}\nmask_dir={mask_dir}")
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None:  raise RuntimeError(f"Failed to read image: {img_path}")
        if mask is None: raise RuntimeError(f"Failed to read mask: {mask_path}")

        mask = (mask > 127).astype(np.uint8)  # 0/1

        img  = img[..., None]   # H W 1
        mask = mask[..., None]  # H W 1

        if self.transform:
            out = self.transform(image=img, mask=mask)
            img_t  = out["image"]  # C,H,W float
            mask_t = out["mask"]   # H,W,1 (albumentations) or tensor

            # make mask tensor 1,H,W
            if isinstance(mask_t, torch.Tensor):
                if mask_t.ndim == 2:
                    mask_t = mask_t.unsqueeze(0)
                elif mask_t.ndim == 3:
                    mask_t = mask_t.permute(2,0,1)
                mask_t = mask_t.float()
                mask_np = mask_t[0].cpu().numpy().astype(np.uint8)
            else:
                # numpy case (rare with ToTensorV2 in place)
                mask_np = mask_t[..., 0].astype(np.uint8)
                mask_t = torch.from_numpy(mask_t).permute(2,0,1).float()
        else:
            img_t  = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
            mask_t = torch.from_numpy(mask.transpose(2,0,1)).float()
            mask_np = mask[..., 0].astype(np.uint8)

        sdf_np = signed_distance_map(mask_np)               # H,W float32 in [-1,1]
        sdf_t  = torch.from_numpy(sdf_np).unsqueeze(0)      # 1,H,W float32

        return img_t, mask_t, sdf_t



# -------------------------
# Resize (divisible by 32)
# -------------------------
H, W = 704, 512

train_tf = A.Compose([
    A.RandomCrop(512,512),     # 🔥 key improvement
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=10,
        p=0.7,
        border_mode=cv2.BORDER_CONSTANT
    ),
    A.OneOf([
        A.GaussNoise(p=1.0),
        A.MotionBlur(p=1.0),
        A.MedianBlur(blur_limit=3, p=1.0),
    ], p=0.3),
    A.RandomBrightnessContrast(p=0.5),
    A.ElasticTransform(alpha=20, sigma=5, alpha_affine=5, p=0.3),
    A.Normalize(mean=(0.0,), std=(1.0,)),
    ToTensorV2()
])


val_tf = A.Compose([
    A.Resize(H, W),
    A.Normalize(mean=(0.0,), std=(1.0,)),
    ToTensorV2()
])


# -------------------------
# Datasets / Loaders
# -------------------------

train_ds = VeinSegDatasetSDF(img_dir, train_msk_dir, transform=train_tf)
val_ds   = VeinSegDatasetSDF(img_dir, val_msk_dir,   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("Train:", len(train_ds), " Val:", len(val_ds))



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

bce = nn.BCEWithLogitsLoss()

def tversky_loss(logits, targets, alpha=0.3, beta=0.7, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(1,2,3))
    fp = (probs * (1 - targets)).sum(dim=(1,2,3))
    fn = ((1 - probs) * targets).sum(dim=(1,2,3))
    t = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - t.mean()

def sdf_reg_loss(sdf_pred, sdf_true):
    # smooth L1 is stable for regression
    return F.smooth_l1_loss(sdf_pred, sdf_true)

LAMBDA_SDF = 0.3  # start with 0.2~0.5

def total_loss(outputs, targets, sdf_true):
    seg_logits = outputs[:, :1]   # (B,1,H,W)
    sdf_pred   = outputs[:, 1:]   # (B,1,H,W)

    L_seg = 0.5 * bce(seg_logits, targets) + 0.5 * tversky_loss(seg_logits, targets)
    L_sdf = sdf_reg_loss(sdf_pred, sdf_true)

    return L_seg + LAMBDA_SDF * L_sdf





In [ ]:
import numpy as np

use_amp = (device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def train_one_epoch(thresh=0.5):
    model.train()
    losses, ious, dices = [], [], []

    for imgs, masks, sdf in train_loader:
        imgs  = imgs.to(device).float()
        masks = masks.to(device).float()
        sdf   = sdf.to(device).float()

        opt.zero_grad(set_to_none=True)

        if use_amp:
            with torch.cuda.amp.autocast():
                outputs = model(imgs)  # (B,2,H,W)
                loss = total_loss(outputs, masks, sdf)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            outputs = model(imgs)
            loss = total_loss(outputs, masks, sdf)
            loss.backward()
            opt.step()

        losses.append(float(loss.detach().cpu()))
        with torch.no_grad():
            ious.append(float(iou_score_hard(outputs, masks, thresh=thresh).detach().cpu()))
            dices.append(float(dice_score_hard(outputs, masks, thresh=thresh).detach().cpu()))

    return float(np.mean(losses)), float(np.mean(ious)), float(np.mean(dices))


@torch.no_grad()
def validate(thresh=0.5):
    model.eval()
    losses, ious, dices = [], [], []

    for imgs, masks, sdf in val_loader:
        imgs  = imgs.to(device).float()
        masks = masks.to(device).float()
        sdf   = sdf.to(device).float()

        outputs = model(imgs)
        loss = total_loss(outputs, masks, sdf)

        losses.append(float(loss.detach().cpu()))
        ious.append(float(iou_score_hard(outputs, masks, thresh=thresh).detach().cpu()))
        dices.append(float(dice_score_hard(outputs, masks, thresh=thresh).detach().cpu()))

    return float(np.mean(losses)), float(np.mean(ious)), float(np.mean(dices))



@torch.no_grad()
def eval_loader(loader, thresh=0.5):
    model.eval()
    losses, ious, dices = [], [], []
    for imgs, masks in loader:
        imgs  = imgs.to(device).float()
        masks = masks.to(device).float()
        if imgs.max() > 1.5:
            imgs = imgs / 255.0
        logits = model(imgs)
        losses.append(total_loss(logits, masks).item())
        ious.append(iou_score_hard(logits, masks, thresh=thresh).item())
        dices.append(dice_score_hard(logits, masks, thresh=thresh).item())
    return float(np.mean(losses)), float(np.mean(ious)), float(np.mean(dices))


In [ ]:
@torch.no_grad()
def dice_score_hard(outputs, targets, thresh=0.5, eps=1e-6):
    seg_logits = outputs[:, :1]
    probs = torch.sigmoid(seg_logits)
    preds = (probs > thresh).float()
    inter = (preds * targets).sum(dim=(1,2,3))
    denom = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    return ((2*inter + eps) / (denom + eps)).mean()

@torch.no_grad()
def iou_score_hard(outputs, targets, thresh=0.5, eps=1e-6):
    seg_logits = outputs[:, :1]
    probs = torch.sigmoid(seg_logits)
    preds = (probs > thresh).float()
    inter = (preds * targets).sum(dim=(1,2,3))
    union = (preds + targets - preds*targets).sum(dim=(1,2,3))
    return ((inter + eps) / (union + eps)).mean()


In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",     # recommended
    encoder_weights="imagenet",
    in_channels=1,
    classes=2                    # <-- 2 channels now
).to(device)


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)



In [ ]:
import numpy as np
import cv2

def signed_distance_map(mask01: np.ndarray) -> np.ndarray:
    """
    mask01: HxW uint8 or bool, values 0/1
    returns: HxW float32 signed distance (positive inside, negative outside), normalized to [-1, 1]
    """
    mask01 = (mask01 > 0).astype(np.uint8)

    # distance inside object
    dist_in = cv2.distanceTransform(mask01, distanceType=cv2.DIST_L2, maskSize=3)

    # distance outside object
    dist_out = cv2.distanceTransform(1 - mask01, distanceType=cv2.DIST_L2, maskSize=3)

    sdf = dist_in - dist_out  # signed distance

    # normalize to [-1, 1] to stabilize regression
    m = np.max(np.abs(sdf))
    if m > 0:
        sdf = sdf / m
    sdf = np.clip(sdf, -1.0, 1.0).astype(np.float32)

    return sdf


In [ ]:
train_dice_history = []
val_dice_history = []
train_loss_history = []
val_loss_history = []


In [ ]:
import torch
import time

EPOCHS = 200
best_val_dice = -1.0
patience = 30
bad_epochs = 0

train_dice_history = []
val_dice_history = []
train_loss_history = []
val_loss_history = []

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    tr_loss, tr_iou, tr_dice = train_one_epoch(thresh=0.5)
    va_loss, va_iou, va_dice = validate(thresh=0.5)

    scheduler.step()

    # Store history
    train_dice_history.append(tr_dice)
    val_dice_history.append(va_dice)
    train_loss_history.append(tr_loss)
    val_loss_history.append(va_loss)

    dt = time.time() - t0

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Dice: {tr_dice:.4f} | "
        f"Val Dice: {va_dice:.4f} | "
        f"Time: {dt:.1f}s"
    )

    # Save best model
    if va_dice > best_val_dice:
        best_val_dice = va_dice
        bad_epochs = 0
        torch.save(model.state_dict(), "/content/best_model.pt")
        print("✅ Saved best model")
    else:
        bad_epochs += 1

    # Early stopping
    if bad_epochs >= patience:
        print("⏹ Early stopping triggered.")
        break

print("\nBest validation Dice:", best_val_dice)




In [ ]:
import os

print(os.listdir("/content"))


In [ ]:
import os

root = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"
print("Inside root:", os.listdir(root)[:50])


In [ ]:
import os, glob

test_img_dir = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"
test_msk_dir = "/content/test_masks"

imgs = sorted([os.path.basename(p) for p in glob.glob(os.path.join(test_img_dir, "*.png"))])
msks = set(os.listdir(test_msk_dir))

matched = [n for n in imgs if n in msks]

print("Total images:", len(imgs))
print("Total masks :", len(msks))
print("Matched pairs:", len(matched))
print("Example match:", matched[:5])
print("Example image only:", [n for n in imgs if n not in msks][:5])
print("Example mask only :", [n for n in msks if n not in set(imgs)][:5])


In [ ]:
from torch.utils.data import DataLoader

test_ds = VeinSegDatasetSDF(test_img_dir, test_msk_dir, transform=val_tf)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test samples:", len(test_ds))
print("Test batches:", len(test_loader))


In [ ]:
import torch

model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
model.to(device)
model.eval()
print("Loaded /content/best_model.pt")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def show_test_preds(loader, thresh=0.8, n_show=8):
    model.eval()
    shown = 0

    for imgs, masks in loader:
        imgs = imgs.to(device).float()
        masks = masks.to(device).float()

        with torch.no_grad():
            logits = model(imgs)
            probs = torch.sigmoid(logits)
            preds = (probs > thresh).float()

        # move to cpu for plotting
        imgs_cpu  = imgs.detach().cpu().numpy()
        masks_cpu = masks.detach().cpu().numpy()
        probs_cpu = probs.detach().cpu().numpy()
        preds_cpu = preds.detach().cpu().numpy()

        b = imgs_cpu.shape[0]
        for i in range(b):
            img  = imgs_cpu[i, 0]
            gt   = masks_cpu[i, 0]
            pr   = preds_cpu[i, 0]
            pb   = probs_cpu[i, 0]

            plt.figure(figsize=(16,4))

            plt.subplot(1,5,1)
            plt.title("Input")
            plt.imshow(img, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,2)
            plt.title("GT")
            plt.imshow(gt, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,3)
            plt.title("Prob map")
            plt.imshow(pb, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,4)
            plt.title(f"Pred (t={thresh})")
            plt.imshow(pr, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,5)
            plt.title("Overlay")
            plt.imshow(img, cmap="gray")
            plt.imshow(pr, alpha=0.35)
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            shown += 1
            if shown >= n_show:
                return

show_test_preds(test_loader, thresh=0.8, n_show=9)


In [ ]:
show_test_preds(test_loader, thresh=0.8, n_show=9)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_dice_history, label="Train Dice")
plt.plot(val_dice_history, label="Validation Dice")

plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Train vs Validation Dice")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def sdf_to_skeleton(sdf, mask=None, eps=0.05):
    """
    sdf: (H,W) float in [-1,1]
    mask: (H,W) optional 0/1
    returns skeleton-like boolean map where |sdf| is small (near center boundary between +/-)
    For vessels, the ridge near zero often tracks a centerline-ish path.
    """
    sk = (np.abs(sdf) < eps)
    if mask is not None:
        sk = sk & (mask > 0.5)   # keep only inside GT mask (optional)
    return sk.astype(np.uint8)

def plot_sdf_regression(loader, thresh=0.8, n_show=8, eps_skel=0.05, title_prefix=""):
    model.eval()
    shown = 0

    for imgs, masks, sdf_gt in loader:
        imgs_d  = imgs.to(device).float()
        masks_d = masks.to(device).float()
        sdf_gt_d = sdf_gt.to(device).float()

        with torch.no_grad():
            out = model(imgs_d)              # (B,2,H,W)
            seg_logits = out[:, :1]
            sdf_pred   = out[:, 1:]          # (B,1,H,W)

            probs = torch.sigmoid(seg_logits)
            preds = (probs > thresh).float()

        # move to cpu numpy
        imgs_np   = imgs_d.detach().cpu().numpy()
        masks_np  = masks_d.detach().cpu().numpy()
        probs_np  = probs.detach().cpu().numpy()
        preds_np  = preds.detach().cpu().numpy()
        sdf_gt_np = sdf_gt_d.detach().cpu().numpy()
        sdf_pr_np = sdf_pred.detach().cpu().numpy()

        B = imgs_np.shape[0]
        for i in range(B):
            img  = imgs_np[i,0]
            gt   = masks_np[i,0]
            pb   = probs_np[i,0]
            pr   = preds_np[i,0]
            sdg  = sdf_gt_np[i,0]   # GT SDF in [-1,1]
            sdp  = sdf_pr_np[i,0]   # Pred SDF (should learn [-1,1])

            # skeleton-like from GT and Pred (two variants)
            sk_gt = sdf_to_skeleton(sdg, mask=gt, eps=eps_skel)
            sk_pr = sdf_to_skeleton(sdp, mask=pr, eps=eps_skel)  # use pred mask for pred skeleton

            plt.figure(figsize=(18,4))
            plt.suptitle(f"{title_prefix} sample {shown+1}  (th={thresh}, sk_eps={eps_skel})", y=1.02)

            plt.subplot(1,5,1)
            plt.title("Input")
            plt.imshow(img, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,2)
            plt.title("GT mask")
            plt.imshow(gt, cmap="gray")
            plt.axis("off")

            plt.subplot(1,5,3)
            plt.title("GT SDF (signed distance)")
            plt.imshow(sdg, cmap="seismic", vmin=-1, vmax=1)  # red/blue signed
            plt.axis("off")

            plt.subplot(1,5,4)
            plt.title("Pred SDF (regression)")
            plt.imshow(sdp, cmap="seismic", vmin=-1, vmax=1)
            plt.axis("off")

            plt.subplot(1,5,5)
            plt.title("Overlay: pred mask + skeleton")
            plt.imshow(img, cmap="gray")
            plt.imshow(pr, alpha=0.30)          # pred mask overlay
            plt.imshow(sk_pr, alpha=0.70)       # skeleton-like overlay
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            # optional: also show GT skeleton overlay if you want
            # plt.figure(figsize=(6,4))
            # plt.title("GT skeleton overlay")
            # plt.imshow(img, cmap="gray")
            # plt.imshow(gt, alpha=0.30)
            # plt.imshow(sk_gt, alpha=0.70)
            # plt.axis("off")
            # plt.show()

            shown += 1
            if shown >= n_show:
                return

# Example usage:
plot_sdf_regression(train_loader, thresh=0.8, n_show=20, eps_skel=0.05, title_prefix="TRAIN")
# plot_sdf_regression(val_loader,   thresh=0.8, n_show=10, eps_skel=0.05, title_prefix="VAL")
# plot_sdf_regression(test_loader,  thresh=0.8, n_show=10, eps_skel=0.05, title_prefix="TEST")


In [ ]:
import torch

model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
model.to(device)
model.eval()

print("SDF model loaded.")


In [ ]:
@torch.no_grad()
def eval_sdf_model(loader, thresh=0.5):
    model.eval()
    losses, ious, dices = [], [], []

    for imgs, masks in loader:   # if using normal dataset
        imgs  = imgs.to(device).float()
        masks = masks.to(device).float()

        out = model(imgs)
        seg_logits = out[:, :1]  # segmentation channel

        probs = torch.sigmoid(seg_logits)
        preds = (probs > thresh).float()

        # Dice
        inter = (preds * masks).sum(dim=(1,2,3))
        denom = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
        dice = (2*inter + 1e-6) / (denom + 1e-6)

        # IoU
        union = (preds + masks - preds*masks).sum(dim=(1,2,3))
        iou = (inter + 1e-6) / (union + 1e-6)

        dices.append(dice.mean().item())
        ious.append(iou.mean().item())

    return float(np.mean(ious)), float(np.mean(dices))


In [ ]:
import numpy as np

best_t = 0
best_d = 0

print("Test threshold sweep:")
for t in np.linspace(0.3, 0.9, 13):
    iou, dice = eval_sdf_model(test_loader, thresh=float(t))
    print(f"t={t:.2f}  dice={dice:.4f}  iou={iou:.4f}")

    if dice > best_d:
        best_d = dice
        best_t = t

print("\nBest threshold:", best_t)
print("Best TEST Dice:", best_d)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.morphology import skeletonize

def mask_to_skeleton(mask01):
    return skeletonize(mask01.astype(bool)).astype(np.uint8)

@torch.no_grad()
def visualize_sdf_on_test(test_loader, thresh=0.9, n_show=9, show_gt=False):
    model.eval()
    shown = 0

    for imgs, masks in test_loader:
        imgs_d = imgs.to(device).float()
        masks_d = masks.to(device).float()

        out = model(imgs_d)               # (B,2,H,W)
        seg_logits = out[:, :1]           # seg channel
        sdf_pred   = out[:, 1:]           # sdf channel (optional to show)

        probs = torch.sigmoid(seg_logits) # (B,1,H,W)
        preds = (probs > thresh).float()

        # to cpu
        imgs_np  = imgs_d.cpu().numpy()
        probs_np = probs.cpu().numpy()
        preds_np = preds.cpu().numpy()
        sdf_np   = sdf_pred.cpu().numpy()
        masks_np = masks_d.cpu().numpy()

        B = imgs_np.shape[0]
        for i in range(B):
            img = imgs_np[i,0]
            pb  = probs_np[i,0]
            pr  = preds_np[i,0]
            sd  = sdf_np[i,0]
            gt  = masks_np[i,0]

            sk = mask_to_skeleton(pr.astype(np.uint8))

            # layout: 4 or 5 panels
            if show_gt:
                plt.figure(figsize=(18,4))
                plt.subplot(1,5,1); plt.title("Input"); plt.imshow(img, cmap="gray"); plt.axis("off")
                plt.subplot(1,5,2); plt.title("Prob map"); plt.imshow(pb, cmap="gray"); plt.axis("off")
                plt.subplot(1,5,3); plt.title(f"Pred mask (t={thresh})"); plt.imshow(pr, cmap="gray"); plt.axis("off")
                plt.subplot(1,5,4); plt.title("Pred skeleton"); plt.imshow(sk, cmap="gray"); plt.axis("off")
                plt.subplot(1,5,5); plt.title("GT mask"); plt.imshow(gt, cmap="gray"); plt.axis("off")
            else:
                plt.figure(figsize=(16,4))
                plt.subplot(1,4,1); plt.title("Input"); plt.imshow(img, cmap="gray"); plt.axis("off")
                plt.subplot(1,4,2); plt.title("Prob map"); plt.imshow(pb, cmap="gray"); plt.axis("off")
                plt.subplot(1,4,3); plt.title(f"Pred mask (t={thresh})"); plt.imshow(pr, cmap="gray"); plt.axis("off")
                plt.subplot(1,4,4); plt.title("Overlay: prob + skeleton")
                plt.imshow(img, cmap="gray")
                plt.imshow(pb, alpha=0.35)
                plt.imshow(sk, alpha=0.9)
                plt.axis("off")

            plt.tight_layout()
            plt.show()

            # optional: show SDF regression map too
            plt.figure(figsize=(6,4))
            plt.title("Pred SDF (regression head)")
            plt.imshow(sd, cmap="seismic", vmin=-1, vmax=1)
            plt.axis("off")
            plt.show()

            shown += 1
            if shown >= n_show:
                return

# Run on test
visualize_sdf_on_test(test_loader, thresh=0.9, n_show=9, show_gt=False)


In [ ]:
import os

UNLABELED_IMG_DIR = "/content/unlabeled_images"

os.makedirs(UNLABELED_IMG_DIR, exist_ok=True)

print("Created folder:", UNLABELED_IMG_DIR)
print("Current folders in /content:")
print(os.listdir("/content"))


In [ ]:
import os

files = os.listdir(UNLABELED_IMG_DIR)
print("Total files:", len(files))

# check duplicate filenames
unique_files = set(files)
print("Unique filenames:", len(unique_files))

if len(files) != len(unique_files):
    print("⚠ Duplicate filenames detected")
else:
    print("✅ No duplicate filenames")


In [ ]:
import hashlib
import glob
import cv2

def image_hash(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return hashlib.md5(img.tobytes()).hexdigest()

hash_map = {}
duplicates = []

for p in glob.glob(os.path.join(UNLABELED_IMG_DIR, "*")):
    h = image_hash(p)
    if h is None:
        continue
    if h in hash_map:
        duplicates.append(p)
    else:
        hash_map[h] = p

print("Duplicate image content found:", len(duplicates))


In [ ]:
LABELED_IMG_DIR = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"


In [ ]:
labeled_hashes = set()
for p in glob.glob(os.path.join(LABELED_IMG_DIR, "*")):
    h = image_hash(p)
    if h:
        labeled_hashes.add(h)

overlap = []
for p in glob.glob(os.path.join(UNLABELED_IMG_DIR, "*")):
    h = image_hash(p)
    if h in labeled_hashes:
        overlap.append(p)

print("Images overlapping with labeled set:", len(overlap))


In [ ]:
for p in overlap:
    os.remove(p)

print("Removed overlaps with labeled set.")
print("Remaining unlabeled images:", len(os.listdir(UNLABELED_IMG_DIR)))


In [ ]:
import os, glob
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

# ✅ CHANGE THESE
UNLABELED_IMG_DIR = "/content/unlabeled_images"   # folder with 200 images
PSEUDO_MASK_DIR   = "/content/pseudo_masks"      # output folder

os.makedirs(PSEUDO_MASK_DIR, exist_ok=True)

# Your trained resolution
H, W = 704, 512

# Threshold you said works best
THRESH = 0.90


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# make sure your model is already created above this cell
# model = smp.Unet(... classes=2 ...)  # your SDF model

ckpt_path = "/content/best_model.pt"   # change if needed
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.to(device)
model.eval()

print("Loaded:", ckpt_path)


In [ ]:
from skimage.morphology import skeletonize
from scipy.ndimage import binary_dilation

def preprocess_gray(path, H, W):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError(f"Failed to read: {path}")
    img = cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA)
    img_f = img.astype(np.float32) / 255.0
    # normalize mean=0 std=1 (your A.Normalize(0,1) does nothing)
    x = torch.from_numpy(img_f[None, None, ...])  # 1,1,H,W
    return img, x

def remove_small_components(mask01, min_area=200):
    """mask01 uint8 {0,1} -> cleaned mask01"""
    mask01 = (mask01 > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask01, connectivity=8)
    out = np.zeros_like(mask01)
    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == k] = 1
    return out

def compute_confidence(prob, mask01):
    """prob float [0..1], mask01 {0,1}"""
    area = mask01.sum()
    if area == 0:
        return 0.0
    return float(prob[mask01 > 0].mean())

def compute_area_fraction(mask01):
    return float(mask01.mean())  # 0..1 of pixels

def pred_skeleton(mask01, thickness=2):
    sk = skeletonize(mask01.astype(bool)).astype(np.uint8)
    if thickness > 0:
        sk = binary_dilation(sk, iterations=thickness).astype(np.uint8)
    return sk


In [ ]:
import pandas as pd

# --- filtering knobs (tune these) ---
MIN_COMPONENT_AREA = 150     # remove tiny blobs
MIN_AREA_FRAC = 0.002        # too tiny -> reject (0.2% pixels)
MAX_AREA_FRAC = 0.25         # too huge -> reject (25% pixels)
MIN_MEAN_CONF  = 0.85        # mean prob inside mask must be high

# visualization
VIS_EVERY = 20               # show every N images
SKELETON_THICKNESS = 2

img_paths = sorted([p for p in glob.glob(os.path.join(UNLABELED_IMG_DIR, "*")) if os.path.isfile(p)])
print("Unlabeled images found:", len(img_paths))

records = []
saved = 0
rejected = 0

for idx, p in enumerate(img_paths):
    fname = os.path.basename(p)

    # preprocess
    img_u8, x = preprocess_gray(p, H, W)
    x = x.to(device)

    # forward
    with torch.no_grad():
        out = model(x)                 # (1,2,H,W) if SDF model
        seg_logits = out[:, :1]
        sdf_pred   = out[:, 1:]        # optional
        prob = torch.sigmoid(seg_logits)[0,0].detach().cpu().numpy()
        sdf  = sdf_pred[0,0].detach().cpu().numpy()

    # threshold + clean
    mask01 = (prob > THRESH).astype(np.uint8)
    mask01 = remove_small_components(mask01, min_area=MIN_COMPONENT_AREA)

    # filter
    area_frac = compute_area_fraction(mask01)
    mean_conf = compute_confidence(prob, mask01)

    keep = (mask01.sum() > 0) and (MIN_AREA_FRAC <= area_frac <= MAX_AREA_FRAC) and (mean_conf >= MIN_MEAN_CONF)

    # save if keep
    out_path = os.path.join(PSEUDO_MASK_DIR, fname)
    if keep:
        cv2.imwrite(out_path, (mask01 * 255).astype(np.uint8))
        saved += 1
        status = "kept"
    else:
        rejected += 1
        status = "rejected"

    records.append({
        "file": fname,
        "keep": keep,
        "mean_conf": mean_conf,
        "area_frac": area_frac,
        "saved_path": out_path if keep else ""
    })

    # visualization
    if (idx % VIS_EVERY == 0) or (idx == len(img_paths)-1):
        sk = pred_skeleton(mask01, thickness=SKELETON_THICKNESS)

        plt.figure(figsize=(18,4))
        plt.suptitle(f"[{status}] {fname} | mean_conf={mean_conf:.3f} area_frac={area_frac:.4f}", y=1.02)

        plt.subplot(1,5,1)
        plt.title("Input")
        plt.imshow(img_u8, cmap="gray")
        plt.axis("off")

        plt.subplot(1,5,2)
        plt.title("Prob map")
        plt.imshow(prob, cmap="gray", vmin=0, vmax=1)
        plt.axis("off")

        plt.subplot(1,5,3)
        plt.title(f"Pseudo mask (t={THRESH})")
        plt.imshow(mask01, cmap="gray")
        plt.axis("off")

        plt.subplot(1,5,4)
        plt.title("Pred SDF (optional)")
        plt.imshow(sdf, cmap="seismic")
        plt.axis("off")

        plt.subplot(1,5,5)
        plt.title("Overlay + skeleton")
        plt.imshow(img_u8, cmap="gray")
        plt.imshow(mask01, alpha=0.30)
        plt.imshow(sk, alpha=0.85)
        plt.axis("off")

        plt.tight_layout()
        plt.show()

print("\nPseudo-labeling done.")
print("✅ Saved:", saved)
print("❌ Rejected:", rejected)

df = pd.DataFrame(records)
csv_path = "/content/pseudo_label_report.csv"
df.to_csv(csv_path, index=False)
print("Report saved:", csv_path)


In [ ]:
import random

saved_paths = sorted(glob.glob(os.path.join(PSEUDO_MASK_DIR, "*")))
print("Saved pseudo masks:", len(saved_paths))

for p in random.sample(saved_paths, min(9, len(saved_paths))):
    fname = os.path.basename(p)
    img_path = os.path.join(UNLABELED_IMG_DIR, fname)

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA)
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    m = (m > 127).astype(np.uint8)

    plt.figure(figsize=(10,3))
    plt.suptitle(fname, y=1.02)

    plt.subplot(1,3,1); plt.title("Input"); plt.imshow(img, cmap="gray"); plt.axis("off")
    plt.subplot(1,3,2); plt.title("Pseudo mask"); plt.imshow(m, cmap="gray"); plt.axis("off")
    plt.subplot(1,3,3); plt.title("Overlay"); plt.imshow(img, cmap="gray"); plt.imshow(m, alpha=0.35); plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
import os, glob, random, shutil

UNLABELED_IMG_DIR = "/content/unlabeled_images"

PSEUDO_TRAIN_IMG = "/content/unlabeled_split/train"
PSEUDO_VAL_IMG   = "/content/unlabeled_split/val"
PSEUDO_TEST_IMG  = "/content/unlabeled_split/test"

for d in [PSEUDO_TRAIN_IMG, PSEUDO_VAL_IMG, PSEUDO_TEST_IMG]:
    os.makedirs(d, exist_ok=True)

# collect all images
imgs = sorted(glob.glob(os.path.join(UNLABELED_IMG_DIR, "*")))
random.seed(42)
random.shuffle(imgs)

n = len(imgs)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_imgs = imgs[:n_train]
val_imgs   = imgs[n_train:n_train+n_val]
test_imgs  = imgs[n_train+n_val:]

print("Total:", n)
print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

# copy files into split folders
for p in train_imgs:
    shutil.copy(p, os.path.join(PSEUDO_TRAIN_IMG, os.path.basename(p)))

for p in val_imgs:
    shutil.copy(p, os.path.join(PSEUDO_VAL_IMG, os.path.basename(p)))

for p in test_imgs:
    shutil.copy(p, os.path.join(PSEUDO_TEST_IMG, os.path.basename(p)))

print("Split complete.")


In [ ]:
PSEUDO_TRAIN_MASK = "/content/pseudo_split/train_masks"
PSEUDO_VAL_MASK   = "/content/pseudo_split/val_masks"
PSEUDO_TEST_MASK  = "/content/pseudo_split/test_masks"

for d in [PSEUDO_TRAIN_MASK, PSEUDO_VAL_MASK, PSEUDO_TEST_MASK]:
    os.makedirs(d, exist_ok=True)

print("Mask folders created.")


In [ ]:
UNLABELED_IMG_DIR
PSEUDO_MASK_DIR


In [ ]:
def generate_pseudo_for_folder(img_dir, out_mask_dir,
                               threshold=0.9,
                               min_area=150,
                               min_conf=0.85):

    os.makedirs(out_mask_dir, exist_ok=True)
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*")))

    kept = 0
    rejected = 0

    for p in img_paths:
        fname = os.path.basename(p)

        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img_r = cv2.resize(img, (W, H))
        x = torch.from_numpy(img_r.astype(np.float32)/255.0)[None,None].to(device)

        with torch.no_grad():
            out = model(x)
            seg_logits = out[:, :1]
            prob = torch.sigmoid(seg_logits)[0,0].cpu().numpy()

        mask = (prob > threshold).astype(np.uint8)

        # remove tiny blobs
        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        cleaned = np.zeros_like(mask)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                cleaned[labels == i] = 1

        mean_conf = prob[cleaned > 0].mean() if cleaned.sum()>0 else 0

        if cleaned.sum()>0 and mean_conf >= min_conf:
            cv2.imwrite(os.path.join(out_mask_dir, fname), cleaned*255)
            kept += 1
        else:
            rejected += 1

    print(f"{img_dir} → kept {kept}, rejected {rejected}")


In [ ]:
import os, glob, cv2, random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------- MODEL ----------
ENCODER = "resnet34"     # "resnet18" | "resnet34" | "resnet50"
H, W = 704, 512

# -------- TRAINING ----------
BATCH_SIZE = 8           # 8–12 on A100, 4–8 on T4
LR = 3e-4
EPOCHS = 200
patience = 30
PSEUDO_WEIGHT = 0.6
LAMBDA_SDF = 0.10

# -------- PATHS ----------
REAL_IMG_DIR = "/content/dataset-files-dabbdf31-47db-4c22-929f-d71ec3c33dbf"
REAL_TRAIN_MASK = "/content/train_masks"
REAL_VAL_MASK   = "/content/val_masks"
REAL_TEST_MASK  = "/content/test_masks"

PSEUDO_TRAIN_IMG = "/content/unlabeled_split/train"
PSEUDO_VAL_IMG   = "/content/unlabeled_split/val"
PSEUDO_TEST_IMG  = "/content/unlabeled_split/test"

PSEUDO_TRAIN_MASK = "/content/pseudo_split/train_masks"
PSEUDO_VAL_MASK   = "/content/pseudo_split/val_masks"
PSEUDO_TEST_MASK  = "/content/pseudo_split/test_masks"


In [ ]:
train_tf = A.Compose([
    A.Resize(H, W),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.03,
        scale_limit=0.05,
        rotate_limit=7,
        p=0.5,
        border_mode=cv2.BORDER_CONSTANT
    ),
    A.Normalize(mean=(0.0,), std=(1.0,)),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(H, W),
    A.Normalize(mean=(0.0,), std=(1.0,)),
    ToTensorV2()
])


In [ ]:
import os, glob, cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class VeinSegSDFDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, weight=1.0):
        self.transform = transform
        self.weight = float(weight)

        img_paths = sorted([p for p in glob.glob(os.path.join(img_dir, "*")) if os.path.isfile(p)])
        pairs = []
        for p in img_paths:
            name = os.path.basename(p)
            mp = os.path.join(mask_dir, name)
            if os.path.isfile(mp):
                pairs.append((p, mp))

        if len(pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found:\nimg_dir={img_dir}\nmask_dir={mask_dir}")

        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    @staticmethod
    def sdf_from_mask(mask01):
        mask01 = (mask01 > 0).astype(np.uint8)
        dist_in = cv2.distanceTransform(mask01, cv2.DIST_L2, 3)
        dist_out = cv2.distanceTransform(1 - mask01, cv2.DIST_L2, 3)
        sdf = dist_in - dist_out
        m = np.max(np.abs(sdf))
        if m > 0:
            sdf = sdf / m
        return np.clip(sdf, -1, 1).astype(np.float32)

    def __getitem__(self, idx):
      img_path, mask_path = self.pairs[idx]

      img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
      mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

      if img is None:
          raise RuntimeError(f"Image read failed: {img_path}")
      if mask is None:
          raise RuntimeError(f"Mask read failed: {mask_path}")

      # 🔥 Force consistent size BEFORE augmentation
      img = cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA)
      mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

      mask01 = (mask > 127).astype(np.uint8)

      img_hwc  = img[..., None]        # H,W,1
      mask_hwc = mask01[..., None]     # H,W,1

      if self.transform:
          out = self.transform(image=img_hwc, mask=mask_hwc)
          img_t = out["image"]          # C,H,W
          mask_t = out["mask"]

          if mask_t.ndim == 3:
              mask_t = mask_t.permute(2,0,1)
          elif mask_t.ndim == 2:
              mask_t = mask_t.unsqueeze(0)

          mask_t = mask_t.float()
      else:
          img_t  = torch.from_numpy(img_hwc.transpose(2,0,1)).float() / 255.0
          mask_t = torch.from_numpy(mask_hwc.transpose(2,0,1)).float()

      # SDF from resized mask
      sdf = self.sdf_from_mask(mask01)
      sdf_t = torch.from_numpy(sdf).unsqueeze(0)

      w = torch.tensor(self.weight, dtype=torch.float32)

      return img_t, mask_t, sdf_t, w




In [ ]:
import os

print("Train images:", len(os.listdir("/content/unlabeled_split/train")))
print("Val images:", len(os.listdir("/content/unlabeled_split/val")))
print("Test images:", len(os.listdir("/content/unlabeled_split/test")))


In [ ]:
generate_pseudo_for_folder(
    img_dir="/content/unlabeled_split/train",
    out_mask_dir="/content/pseudo_split/train_masks",
    threshold=0.9
)

generate_pseudo_for_folder(
    img_dir="/content/unlabeled_split/val",
    out_mask_dir="/content/pseudo_split/val_masks",
    threshold=0.9
)

generate_pseudo_for_folder(
    img_dir="/content/unlabeled_split/test",
    out_mask_dir="/content/pseudo_split/test_masks",
    threshold=0.9
)


In [ ]:
real_train = VeinSegSDFDataset(REAL_IMG_DIR, REAL_TRAIN_MASK, train_tf, 1.0)
real_val   = VeinSegSDFDataset(REAL_IMG_DIR, REAL_VAL_MASK,   val_tf,   1.0)
real_test  = VeinSegSDFDataset(REAL_IMG_DIR, REAL_TEST_MASK,  val_tf,   1.0)

pseudo_train = VeinSegSDFDataset(PSEUDO_TRAIN_IMG, PSEUDO_TRAIN_MASK, train_tf, PSEUDO_WEIGHT)
pseudo_val   = VeinSegSDFDataset(PSEUDO_VAL_IMG,   PSEUDO_VAL_MASK,   val_tf,   PSEUDO_WEIGHT)
pseudo_test  = VeinSegSDFDataset(PSEUDO_TEST_IMG,  PSEUDO_TEST_MASK,  val_tf,   PSEUDO_WEIGHT)

train_ds = ConcatDataset([real_train, pseudo_train])
val_ds   = ConcatDataset([real_val,   pseudo_val])
test_ds  = ConcatDataset([real_test,  pseudo_test])

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))


In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(device=="cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=(device=="cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=(device=="cuda"))


In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.nn.functional as F

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights="imagenet",
    in_channels=1,
    classes=2
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

use_amp = (device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


In [ ]:
import torch
import torch.nn as nn

bce = nn.BCEWithLogitsLoss(reduction="none")
smooth_l1 = nn.SmoothL1Loss(reduction="none")  # for SDF

# lighter punish settings
PSEUDO_WEIGHT = 0.5
LAMBDA_SDF = 0.10

def tversky_loss(logits, targets, alpha=0.3, beta=0.7, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs*targets).sum((1,2,3))
    fp = (probs*(1-targets)).sum((1,2,3))
    fn = ((1-probs)*targets).sum((1,2,3))
    return 1 - (tp+eps)/(tp+alpha*fp+beta*fn+eps)   # (B,)

def total_loss(out, mask, sdf, w):
    seg = out[:, :1]
    sdf_pred = out[:, 1:]

    # per-sample seg losses
    L_bce = bce(seg, mask).mean((1,2,3))            # (B,)
    L_tv  = tversky_loss(seg, mask)                 # (B,)

    # per-sample sdf loss (SmoothL1 is gentler)
    L_sdf = smooth_l1(sdf_pred, sdf).mean((1,2,3))  # (B,)

    # ✅ lighter punish: more BCE, less Tversky
    L = 0.6*L_bce + 0.4*L_tv + LAMBDA_SDF*L_sdf

    # sample weighting (pseudo weaker/stronger depending on your setting)
    return (L * w).mean()



In [ ]:
def train_one_epoch():
    model.train()
    losses = []
    for imgs, masks, sdf, w in train_loader:
        imgs, masks, sdf, w = imgs.to(device), masks.to(device), sdf.to(device), w.to(device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(imgs)
            loss = total_loss(out, masks, sdf, w)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        losses.append(loss.item())
    return np.mean(losses)

@torch.no_grad()
def validate():
    model.eval()
    losses = []
    for imgs, masks, sdf, w in val_loader:
        imgs, masks, sdf, w = imgs.to(device), masks.to(device), sdf.to(device), w.to(device)
        out = model(imgs)
        loss = total_loss(out, masks, sdf, w)
        losses.append(loss.item())
    return np.mean(losses)


In [ ]:
import torch

@torch.no_grad()
def dice_hard(logits, targets, thresh=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > thresh).float()
    inter = (preds * targets).sum(dim=(1,2,3))
    denom = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (denom + eps)
    return dice.mean()

@torch.no_grad()
def iou_hard(logits, targets, thresh=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > thresh).float()
    inter = (preds * targets).sum(dim=(1,2,3))
    union = (preds + targets - preds*targets).sum(dim=(1,2,3))
    iou = (inter + eps) / (union + eps)
    return iou.mean()

print("dice_hard and iou_hard defined.")


In [ ]:
import numpy as np
import time
import torch

best_val_dice = -1.0
bad_epochs = 0

train_dice_history = []
val_dice_history = []
train_loss_history = []
val_loss_history = []

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    # ---- TRAIN ----
    model.train()
    train_losses = []
    train_dices = []

    for imgs, masks, sdf, w in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        sdf   = sdf.to(device)
        w     = w.to(device)

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(imgs)
            loss = total_loss(out, masks, sdf, w)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        train_losses.append(loss.item())

        seg_logits = out[:, :1]
        train_dices.append(dice_hard(seg_logits, masks, thresh=0.5).item())

    tr_loss = float(np.mean(train_losses))
    tr_dice = float(np.mean(train_dices))

    # ---- VALIDATE ----
    model.eval()
    val_losses = []
    val_dices = []

    with torch.no_grad():
        for imgs, masks, sdf, w in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            sdf   = sdf.to(device)
            w     = w.to(device)

            out = model(imgs)
            loss = total_loss(out, masks, sdf, w)

            val_losses.append(loss.item())

            seg_logits = out[:, :1]
            val_dices.append(dice_hard(seg_logits, masks, thresh=0.5).item())

    va_loss = float(np.mean(val_losses))
    va_dice = float(np.mean(val_dices))

    scheduler.step()

    # ---- STORE HISTORY ----
    train_loss_history.append(tr_loss)
    val_loss_history.append(va_loss)
    train_dice_history.append(tr_dice)
    val_dice_history.append(va_dice)

    dt = time.time() - t0

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Dice: {tr_dice:.4f} | Val Dice: {va_dice:.4f} | "
        f"Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | "
        f"{dt:.1f}s"
    )

    # ---- EARLY STOP ON DICE ----
    if va_dice > best_val_dice:
        best_val_dice = va_dice
        bad_epochs = 0
        torch.save(model.state_dict(), "/content/best_model_pseudo.pt")
        print("✅ Saved best model (based on Val Dice)")
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("⏹ Early stopping triggered.")
        break

print("\nBest validation Dice:", best_val_dice)



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_dice_history, label="Train Dice")
plt.plot(val_dice_history, label="Validation Dice")

plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Train vs Validation Dice")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import random

def _to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.array(x)

def visualize_samples(model, loader, device, num_samples=5, threshold=0.5):
    model.eval()
    dataset = loader.dataset
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, 0)

    with torch.no_grad():
        for row, idx in enumerate(indices):
            sample = dataset[idx]

            # Handle datasets that return (img, mask, ...) with extras
            if isinstance(sample, (list, tuple)):
                image = sample[0]
                gt = sample[1]
                extras = sample[2:]
            else:
                raise ValueError("Dataset __getitem__ must return a tuple/list like (image, mask, ...)")

            # If gt has multiple channels, assume first channel is the binary mask
            gt_np = _to_numpy(gt)
            if gt_np.ndim == 3:      # (C,H,W)
                gt_mask = gt_np[0]
            elif gt_np.ndim == 2:    # (H,W)
                gt_mask = gt_np
            else:
                # Sometimes gt is stored as (H,W,1) etc.
                gt_mask = np.squeeze(gt_np)

            # Move image to device and make (1, C, H, W)
            if image.ndim == 2:
                image = image.unsqueeze(0)  # (1,H,W)
            image_b = image.unsqueeze(0).to(device)

            # Forward pass (support models returning tuple/dict)
            out = model(image_b)
            if isinstance(out, (tuple, list)):
                logits = out[0]
            elif isinstance(out, dict):
                logits = out.get("logits", next(iter(out.values())))
            else:
                logits = out

            # If logits has >1 channel, assume channel 0 is the main mask
            prob = torch.sigmoid(logits)[0]
            if prob.ndim == 3:  # (C,H,W)
                prob_mask = prob[0].detach().cpu().numpy()
            else:               # (H,W)
                prob_mask = prob.detach().cpu().numpy()

            binary_pred = (prob_mask > threshold).astype(np.float32)

            img_np = _to_numpy(image_b[0])
            if img_np.ndim == 3:
                img_show = img_np[0]  # show first channel
            else:
                img_show = img_np

            # Overlay (green prediction)
            overlay = np.stack([img_show]*3, axis=-1)
            overlay[..., 1] = np.clip(overlay[..., 1] + 0.7 * binary_pred, 0, 1)

            # Optional label type (if your dataset provides it)
            label_type = ""
            for ex in extras:
                # common patterns: bool flag, string, dict with 'is_pseudo'
                if isinstance(ex, (bool, np.bool_)):
                    label_type = "Pseudo" if ex else "Real"
                    break
                if isinstance(ex, str):
                    if "pseudo" in ex.lower():
                        label_type = "Pseudo"
                        break
                    if "real" in ex.lower():
                        label_type = "Real"
                        break
                if isinstance(ex, dict) and "is_pseudo" in ex:
                    label_type = "Pseudo" if ex["is_pseudo"] else "Real"
                    break

            axes[row, 0].imshow(prob_mask, cmap="viridis")
            axes[row, 0].set_title("Probability Mask")

            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title(f"GT Mask {('| '+label_type) if label_type else ''}")

            axes[row, 2].imshow(img_show, cmap="gray")
            axes[row, 2].set_title("Real Image")

            axes[row, 3].imshow(overlay)
            axes[row, 3].set_title("Overlay (Pred)")

            for col in range(4):
                axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()



In [ ]:
visualize_samples(model, val_loader, device, num_samples=5)


In [ ]:
print(type(val_loader.dataset[0]), len(val_loader.dataset[0]))


In [ ]:
import torch
import numpy as np

def dice_score(pred_bin: torch.Tensor, gt_bin: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """
    pred_bin, gt_bin: (B,1,H,W) float/bool tensors in {0,1}
    returns: (B,) dice
    """
    pred_bin = pred_bin.float()
    gt_bin = gt_bin.float()
    inter = (pred_bin * gt_bin).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + gt_bin.sum(dim=(1,2,3))
    return (2 * inter + eps) / (union + eps)


In [ ]:
def evaluate_on_loader(model, loader, device, threshold=0.5, max_batches=None):
    model.eval()
    all_dice = []

    with torch.no_grad():
        for b_idx, batch in enumerate(loader):
            if max_batches is not None and b_idx >= max_batches:
                break

            # Batch can be (img, mask, ...) with extras
            if isinstance(batch, (list, tuple)):
                images = batch[0]
                gts = batch[1]
            else:
                raise ValueError("Loader must yield a tuple/list like (images, masks, ...)")

            images = images.to(device)

            # Ensure GT shape is (B,1,H,W)
            if torch.is_tensor(gts):
                gts_t = gts
            else:
                gts_t = torch.tensor(gts)
            if gts_t.ndim == 3:  # (B,H,W)
                gts_t = gts_t.unsqueeze(1)
            elif gts_t.ndim == 4:
                pass
            else:
                gts_t = gts_t.squeeze()
                if gts_t.ndim == 3:
                    gts_t = gts_t.unsqueeze(1)

            gts_t = gts_t.to(device).float()
            gts_bin = (gts_t > 0.5).float()

            out = model(images)
            # Support models returning tuple/dict
            if isinstance(out, (tuple, list)):
                logits = out[0]
            elif isinstance(out, dict):
                logits = out.get("logits", next(iter(out.values())))
            else:
                logits = out

            probs = torch.sigmoid(logits)

            # If model outputs multiple channels, take channel 0 as main mask
            if probs.ndim == 4 and probs.size(1) > 1:
                probs = probs[:, 0:1, :, :]
            elif probs.ndim == 3:  # (B,H,W)
                probs = probs.unsqueeze(1)

            pred_bin = (probs > threshold).float()

            d = dice_score(pred_bin, gts_bin)  # (B,)
            all_dice.append(d.detach().cpu())

    all_dice = torch.cat(all_dice, dim=0)
    mean_dice = all_dice.mean().item()
    median_dice = all_dice.median().item()
    p10 = torch.quantile(all_dice, 0.10).item()
    p90 = torch.quantile(all_dice, 0.90).item()

    return {
        "mean_dice": mean_dice,
        "median_dice": median_dice,
        "p10": p10,
        "p90": p90,
        "all_dice": all_dice.numpy()
    }


In [ ]:
test_metrics = evaluate_on_loader(model, test_loader, device, threshold=0.5)
print(f"Test Dice (mean):   {test_metrics['mean_dice']:.4f}")
print(f"Test Dice (median): {test_metrics['median_dice']:.4f}")
print(f"Test Dice (p10/p90): {test_metrics['p10']:.4f} / {test_metrics['p90']:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import random

def _as_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.array(x)

def visualize_test_samples(model, loader, device, num_samples=5, threshold=0.5):
    model.eval()
    ds = loader.dataset
    idxs = random.sample(range(len(ds)), min(num_samples, len(ds)))

    fig, axes = plt.subplots(len(idxs), 4, figsize=(16, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = np.expand_dims(axes, 0)

    with torch.no_grad():
        for r, idx in enumerate(idxs):
            sample = ds[idx]
            if not isinstance(sample, (list, tuple)) or len(sample) < 2:
                raise ValueError("Dataset item must be like (image, mask, ...)")

            img = sample[0]
            gt = sample[1]
            extras = sample[2:]

            # image -> (1,C,H,W)
            if img.ndim == 2:
                img = img.unsqueeze(0)  # (1,H,W)
            img_b = img.unsqueeze(0).to(device)

            # gt -> (1,1,H,W)
            gt_np = _as_numpy(gt)
            if gt_np.ndim == 3:        # (C,H,W)
                gt_mask = gt_np[0]
            elif gt_np.ndim == 2:
                gt_mask = gt_np
            else:
                gt_mask = np.squeeze(gt_np)

            gt_t = torch.tensor(gt_mask, device=device).float().unsqueeze(0).unsqueeze(0)
            gt_bin = (gt_t > 0.5).float()

            out = model(img_b)
            if isinstance(out, (tuple, list)):
                logits = out[0]
            elif isinstance(out, dict):
                logits = out.get("logits", next(iter(out.values())))
            else:
                logits = out

            probs = torch.sigmoid(logits)

            # take channel 0 if multi-channel
            if probs.ndim == 4 and probs.size(1) > 1:
                probs = probs[:, 0:1, :, :]

            prob = probs[0, 0].detach().cpu().numpy()
            pred_bin = (prob > threshold).astype(np.float32)

            # per-sample dice
            pred_t = torch.tensor(pred_bin, device=device).unsqueeze(0).unsqueeze(0)
            d = dice_score(pred_t, gt_bin).item()

            img_np = _as_numpy(img_b[0])
            img_show = img_np[0] if img_np.ndim == 3 else img_np

            overlay = np.stack([img_show]*3, axis=-1)
            overlay[..., 1] = np.clip(overlay[..., 1] + 0.7 * pred_bin, 0, 1)

            # Optional: label type if dataset provides it
            label_type = ""
            for ex in extras:
                if isinstance(ex, (bool, np.bool_)):
                    label_type = "Pseudo" if ex else "Real"
                    break
                if isinstance(ex, str):
                    if "pseudo" in ex.lower():
                        label_type = "Pseudo"; break
                    if "real" in ex.lower():
                        label_type = "Real"; break
                if isinstance(ex, dict) and "is_pseudo" in ex:
                    label_type = "Pseudo" if ex["is_pseudo"] else "Real"
                    break

            axes[r, 0].imshow(prob, cmap="viridis")
            axes[r, 0].set_title("Probability Mask")

            axes[r, 1].imshow(gt_mask, cmap="gray")
            axes[r, 1].set_title(f"GT Mask{(' | '+label_type) if label_type else ''}")

            axes[r, 2].imshow(img_show, cmap="gray")
            axes[r, 2].set_title(f"Test Image | Dice={d:.3f}")

            axes[r, 3].imshow(overlay)
            axes[r, 3].set_title("Overlay (Pred)")

            for c in range(4):
                axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_test_samples(model, test_loader, device, num_samples=20, threshold=0.5)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random

def _skeletonize(binary_mask_np):
    binary = (binary_mask_np > 0).astype(np.uint8)
    try:
        from skimage.morphology import skeletonize
        return skeletonize(binary.astype(bool)).astype(np.uint8)
    except Exception:
        return binary  # fallback

def visualize_sdf_skeleton_overlay(model, loader, device, num_samples=5,
                                   threshold=0.5, eps_sdf=0.05):
    """
    Panels:
    Image | Pred SDF | Skeleton | Skeleton overlay

    Assumes SDF is normalized in [-1,1]
    """
    model.eval()
    ds = loader.dataset
    idxs = random.sample(range(len(ds)), min(num_samples, len(ds)))

    fig, axes = plt.subplots(len(idxs), 4, figsize=(18, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = np.expand_dims(axes, 0)

    with torch.no_grad():
        for r, idx in enumerate(idxs):
            sample = ds[idx]
            img = sample[0]

            if img.ndim == 2:
                img = img.unsqueeze(0)
            img_b = img.unsqueeze(0).to(device)

            out = model(img_b)
            if isinstance(out, (tuple, list)):
                logits = out[0]
            elif isinstance(out, dict):
                logits = out.get("logits", next(iter(out.values())))
            else:
                logits = out

            seg_logits = logits[:, 0:1]
            sdf_pred   = logits[:, 1:2]

            seg_prob = torch.sigmoid(seg_logits)[0, 0].cpu().numpy()
            sdf_map  = sdf_pred[0, 0].cpu().numpy()   # already in [-1,1]

            # Skeleton from SDF zero-band
            zero_band = (np.abs(sdf_map) < eps_sdf).astype(np.uint8)

            # Optional: restrict to confident vessel region
            pred_mask = (seg_prob > threshold).astype(np.uint8)
            zero_band = zero_band * pred_mask

            skel = _skeletonize(zero_band)

            img_np = img_b[0].cpu().numpy()
            img_show = img_np[0] if img_np.ndim == 3 else img_np

            # Overlay skeleton in red
            overlay = np.stack([img_show]*3, axis=-1)
            overlay[..., 0] = np.clip(overlay[..., 0] + 0.9 * skel, 0, 1)

            # Plot
            axes[r, 0].imshow(img_show, cmap="gray")
            axes[r, 0].set_title("Actual Image")
            axes[r, 0].axis("off")

            axes[r, 1].imshow(sdf_map, cmap="seismic", vmin=-1, vmax=1)
            axes[r, 1].set_title("Pred SDF [-1,1]")
            axes[r, 1].axis("off")

            axes[r, 2].imshow(skel, cmap="gray")
            axes[r, 2].set_title("Skeleton (SDF zero-band)")
            axes[r, 2].axis("off")

            axes[r, 3].imshow(overlay)
            axes[r, 3].set_title("Skeleton Overlay")
            axes[r, 3].axis("off")

    plt.tight_layout()
    plt.show()




In [ ]:
visualize_sdf_skeleton_overlay(model, test_loader, device, num_samples=5)



In [ ]:
import os

same_hand_test = "/content/same_hand_test"

os.makedirs(same_hand_test, exist_ok=True)

print("Created folder:", same_hand_test)
print("Current folders in /content:")
print(os.listdir("/content"))

In [ ]:
# ------------------ FULL PIPELINE CELL: inference + metrics + visualization ------------------
import os, glob, json, csv
from pathlib import Path
import numpy as np
import cv2
import torch
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from itertools import combinations
import matplotlib.pyplot as plt

# ------------------ USER CONFIG (edit only these) ------------------
CKPT_PATH = "/content/best_model_pseudo.pt"   # path to checkpoint
FOLDER    = "/content/same_hand_test"         # folder containing your 4 original images
OUT_DIR   = "/content/same_hand_test_out"     # where outputs will be written
H, W      = 704, 512                          # model resolution (H,W)
ENCODER   = "resnet34"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
THRESH    = 0.5
SAVE_PLOTS = True
SHOW_SKELETON = True
# ------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# ------------------ Build model and load checkpoint ------------------
model = smp.Unet(encoder_name=ENCODER, encoder_weights="imagenet", in_channels=1, classes=2)
state = torch.load(CKPT_PATH, map_location="cpu")
# handle either raw state_dict or saved dict containing "state_dict"
if isinstance(state, dict) and "state_dict" in state:
    sd = state["state_dict"]
else:
    sd = state
# strip "module." if present
new_sd = {}
for k, v in sd.items():
    new_k = k.replace("module.", "") if k.startswith("module.") else k
    new_sd[new_k] = v
model.load_state_dict(new_sd)
model.to(DEVICE)
model.eval()

# ------------------ Validation transform (same as training val_tf) ------------------
val_tf = A.Compose([
    A.Resize(H, W),
    A.Normalize(mean=(0.0,), std=(1.0,)),
    ToTensorV2()
])

# helper preprocess that returns a tensor (model resolution) and original raw grayscale
def preprocess_cv2(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError(f"Failed to read {path}")
    out = val_tf(image=img)
    tensor = out["image"].unsqueeze(0).to(DEVICE)   # [1,1,H,W]
    return tensor, img

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def dice_np(a, b, eps=1e-8):
    a = (a > 0).astype(np.uint8)
    b = (b > 0).astype(np.uint8)
    if a.shape != b.shape:
        h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
        a = cv2.resize(a, (w,h), interpolation=cv2.INTER_NEAREST)
        b = cv2.resize(b, (w,h), interpolation=cv2.INTER_NEAREST)
    inter = np.logical_and(a,b).sum()
    return (2.0 * inter) / (a.sum() + b.sum() + eps)

# ------------------ Inference loop: save model-resolution and orig-resolution overlays ------------------
img_paths = sorted([p for p in glob.glob(os.path.join(FOLDER, "*")) if os.path.isfile(p)])
if len(img_paths) == 0:
    raise RuntimeError(f"No images found in {FOLDER}")

results = []
print(f"Running inference on {len(img_paths)} images...")

for p in img_paths:
    tensor, orig_raw = preprocess_cv2(p)   # tensor is resized to (H,W); orig_raw is original raw image
    with torch.no_grad():
        out = model(tensor)  # [1, classes, H, W]
    # segmentation logits channel 0
    logits = out.cpu().numpy()[0, 0, :, :]   # shape (H, W)
    probs = sigmoid_np(logits)
    mask = (probs > THRESH).astype(np.uint8)  # model resolution mask (H,W)
    base = Path(p).stem

    # Save model-resolution prob and mask and overlay (resized orig)
    np.save(os.path.join(OUT_DIR, f"{base}_prob.npy"), probs.astype(np.float32))
    cv2.imwrite(os.path.join(OUT_DIR, f"{base}_mask.png"), (mask*255).astype(np.uint8))

    # create overlay at model resolution (paint red where mask==1)
    orig_resized = cv2.resize(orig_raw, (W, H), interpolation=cv2.INTER_AREA)  # NOTE: cv2 size is (W,H)
    orig_rgb_resized = cv2.cvtColor(orig_resized, cv2.COLOR_GRAY2BGR)
    overlay_resized = orig_rgb_resized.copy()
    overlay_resized[mask == 1] = (0,0,255)   # BGR red
    blended_resized = cv2.addWeighted(orig_rgb_resized, 0.6, overlay_resized, 0.4, 0)
    cv2.imwrite(os.path.join(OUT_DIR, f"{base}_overlay.png"), blended_resized)

    # ALSO save overlay at original raw resolution (upsample mask with INTER_NEAREST)
    mask_up = cv2.resize((mask*255).astype(np.uint8), (orig_raw.shape[1], orig_raw.shape[0]), interpolation=cv2.INTER_NEAREST)
    orig_rgb_raw = cv2.cvtColor(orig_raw, cv2.COLOR_GRAY2BGR)
    overlay_raw = orig_rgb_raw.copy()
    overlay_raw[mask_up > 127] = (0,0,255)
    blended_raw = cv2.addWeighted(orig_rgb_raw, 0.6, overlay_raw, 0.4, 0)
    cv2.imwrite(os.path.join(OUT_DIR, f"{base}_overlay_origsize.png"), blended_raw)

    results.append({"path": p, "base": base, "prob": probs, "mask": mask, "mask_up": mask_up})
    print(f"Saved: {base} (model-res and orig-res overlays)")

# ------------------ Pairwise Dice across saved masks (use model-resolution masks) ------------------
pairs = []
for (i,j) in combinations(range(len(results)), 2):
    m1 = results[i]["mask"]
    m2 = results[j]["mask"]
    d = dice_np(m1, m2)
    pairs.append({"a": results[i]["base"], "b": results[j]["base"], "dice": float(d)})
    print(f"{results[i]['base']} vs {results[j]['base']}: Dice={d:.4f}")

# write CSV of pairwise dice
csv_pairs = os.path.join(OUT_DIR, "pairwise_dice.csv")
with open(csv_pairs, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["a","b","dice"])
    writer.writeheader()
    for p in pairs:
        writer.writerow(p)
print("Pairwise dice saved to", csv_pairs)

# ------------------ Per-segment confidence variation using skeletons (model-resolution) ------------------
conf_stats = []
for r in results:
    base = r["base"]
    mask = r["mask"].astype(np.uint8)
    probs = r["prob"]
    if mask.sum() == 0:
        conf_stats.append({"base": base, "note": "no_mask"})
        continue

    sk = skeletonize((mask > 0).astype(np.uint8))
    lbl = label(sk)
    seg_stats = []
    for region in regionprops(lbl):
        coords = region.coords  # (row, col) pairs
        vals = np.array([probs[y, x] for (y, x) in coords])
        if vals.size == 0:
            continue
        maxmin_rel = float((vals.max() - vals.min()) / (vals.mean() + 1e-8))
        std_rel = float(vals.std() / (vals.mean() + 1e-8))
        seg_stats.append({
            "n_points": int(vals.size),
            "maxmin_rel": maxmin_rel,
            "std_rel": std_rel,
            "mean_conf": float(vals.mean()),
            "min_conf": float(vals.min()),
            "max_conf": float(vals.max())
        })
    conf_stats.append({"base": base, "n_segments": len(seg_stats), "segments": seg_stats})

# save confidence stats JSON
json_conf = os.path.join(OUT_DIR, "confidence_stats.json")
with open(json_conf, "w") as f:
    json.dump(conf_stats, f, indent=2)
print("Per-segment confidence stats saved to", json_conf)

# ------------------ Generate & save per-sample 4-panel PNGs and show inline ------------------
for r in results:
    base = r["base"]
    probs = r["prob"]
    mask = r["mask"]
    mask_up = r["mask_up"]
    # model-res overlay
    overlay_path = os.path.join(OUT_DIR, f"{base}_overlay.png")
    # orig raw path: get from input folder if exists
    orig_path = r["path"]
    orig_raw = cv2.imread(orig_path, cv2.IMREAD_GRAYSCALE)

    # build visuals (use model-res visuals for prob and mask aligned to model resolution)
    prob_vis = (probs*255).astype(np.uint8)
    mask_vis = (mask*255).astype(np.uint8)
    overlay_vis = cv2.cvtColor(cv2.imread(overlay_path), cv2.COLOR_BGR2RGB)

    # create figure
    fig, axs = plt.subplots(1,4, figsize=(16,4))
    im0 = axs[0].imshow(prob_vis, cmap="viridis")
    axs[0].set_title("Probability"); axs[0].axis("off")
    plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

    axs[1].imshow(mask_vis, cmap="gray"); axs[1].set_title("Mask (model res)"); axs[1].axis("off")
    axs[2].imshow(orig_raw, cmap="gray"); axs[2].set_title("Original (raw)"); axs[2].axis("off")

    # overlay with skeleton optionally
    ov = overlay_vis.copy()
    if SHOW_SKELETON and mask.sum() > 0:
        sk = skeletonize((mask > 0).astype(np.uint8))
        ys, xs = np.where(sk)
        for (y,x) in zip(ys, xs):
            if 0 <= y < ov.shape[0] and 0 <= x < ov.shape[1]:
                ov[y, x] = (255,255,255)
    axs[3].imshow(ov); axs[3].set_title("Overlay (model res)"); axs[3].axis("off")

    plt.suptitle(base)
    out_png = os.path.join(PLOTS_DIR, f"{base}_4panel.png")
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved 4-panel for", base, "->", out_png)

print("All done. Outputs saved in:", OUT_DIR)



In [ ]:
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from skimage.morphology import skeletonize


# ---------------------------------------------
# Extract connected skeleton segments
# ---------------------------------------------
def extract_segments(skel):
    visited = np.zeros_like(skel, dtype=bool)
    segments = []

    h, w = skel.shape

    def get_neighbors(y, x):
        neighbors = []
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                if dy == 0 and dx == 0:
                    continue
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w:
                    if skel[ny, nx]:
                        neighbors.append((ny, nx))
        return neighbors

    for y in range(h):
        for x in range(w):
            if skel[y, x] and not visited[y, x]:
                stack = [(y, x)]
                segment = []

                while stack:
                    cy, cx = stack.pop()
                    if visited[cy, cx]:
                        continue
                    visited[cy, cx] = True
                    segment.append((cy, cx))

                    for ny, nx in get_neighbors(cy, cx):
                        if not visited[ny, nx]:
                            stack.append((ny, nx))

                if len(segment) > 5:
                    segments.append(segment)

    return segments


# ---------------------------------------------
# Compute segment metrics (PIXELS)
# ---------------------------------------------
def compute_segment_metrics(segment, dist_map):
    coords = np.array(segment)

    # Geodesic length (pixels)
    diffs = np.diff(coords, axis=0)
    step_lengths = np.sqrt(diffs[:,0]**2 + diffs[:,1]**2)
    path_length_px = np.sum(step_lengths)

    # Euclidean distance
    y0, x0 = coords[0]
    y1, x1 = coords[-1]
    euclidean_px = np.sqrt((y1 - y0)**2 + (x1 - x0)**2)

    straightness = 0
    if path_length_px > 0:
        straightness = (euclidean_px / path_length_px) * 100
        straightness = min(straightness, 100.0)

    # Diameter (pixels)
    radii = [dist_map[y, x] for y, x in segment]
    diameter_px = 2 * np.mean(radii)

    return path_length_px, diameter_px, straightness


# ---------------------------------------------
# Generate unlimited distinct colors
# ---------------------------------------------
def generate_colors(n):
    cmap = cm.get_cmap('hsv', n)
    colors = []
    for i in range(n):
        color = cmap(i)
        rgb = tuple(int(255 * c) for c in color[:3])
        colors.append(rgb)
    return colors


# ---------------------------------------------
# MAIN FUNCTION
# ---------------------------------------------
def batch_analyze_veins(loader, model, num_to_show=5):
    model.eval()
    shown = 0

    for imgs, _, *_ in loader:
        imgs_d = imgs.to(device).float()

        with torch.no_grad():
            out = model(imgs_d)
            seg_logits = out[:, :1] if out.shape[1] > 1 else out
            preds = (torch.sigmoid(seg_logits) > 0.5).float()

        for i in range(len(imgs)):

            img_np = imgs[i, 0].cpu().numpy()
            mask_np = preds[i, 0].cpu().numpy().astype(np.uint8)

            dist_map = cv2.distanceTransform(mask_np, cv2.DIST_L2, 5)
            skeleton = skeletonize(mask_np > 0).astype(np.uint8)
            segments = extract_segments(skeleton)

            print("\n--- New Image ---")

            # Generate unique colors for each segment
            segment_colors = generate_colors(len(segments))

            # Base RGB image (no mask flood)
            rgb_segments = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)

            for idx, seg in enumerate(segments):
                color = segment_colors[idx]

                length_px, diameter_px, straightness = compute_segment_metrics(
                    seg, dist_map
                )

                # Draw thicker segment using circles (radius=2)
                for y, x in seg:
                    cv2.circle(rgb_segments, (x, y), 2, color, -1)

                print(f"Segment {idx+1}: "
                      f"Length={length_px:.2f}px | "
                      f"Diameter={diameter_px:.2f}px | "
                      f"Straightness={straightness:.1f}% | "
                      f"Color={color}")

            # ---------------------------------
            # VISUALIZATION
            # ---------------------------------

            plt.figure(figsize=(18,5))


            plt.subplot(1,4,1)
            plt.imshow(img_np, cmap='gray')
            plt.title("Original Image")
            plt.axis("off")


            plt.subplot(1,4,2)
            plt.imshow(mask_np, cmap='gray')
            plt.title("Predicted Mask")
            plt.axis("off")


            plt.subplot(1,4,3)
            plt.imshow(skeleton, cmap='gray')
            plt.title("Skeleton")
            plt.axis("off")


            plt.subplot(1,4,4)
            plt.imshow(rgb_segments)
            plt.title("Segments")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            shown += 1
            if shown >= num_to_show:
                return


# Run
batch_analyze_veins(test_loader, model)